In [1]:
# Cell 1 — Install dependencies
!pip install -q transformers accelerate bitsandbytes neo4j python-dotenv groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 22.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 8.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 2.4 MB/s eta 0:00:00ta 0:00:01


In [57]:
!pip install -q nltk

In [58]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt')
nltk.download('punkt_tab')

def compute_google_bleu(gold_cypher, pred_cypher):
    """Compute sentence-level Google-BLEU between gold and predicted Cypher"""
    if not pred_cypher:
        return 0.0
    try:
        # tokenize by splitting on whitespace and punctuation
        def tokenize(text):
            return re.findall(r'\w+|[^\w\s]', text.lower())
        
        reference = [tokenize(gold_cypher)]
        hypothesis = tokenize(pred_cypher)
        
        smoothing = SmoothingFunction().method3
        score = sentence_bleu(
            reference,
            hypothesis,
            weights=(0.25, 0.25, 0.25, 0.25),  # 4-gram BLEU
            smoothing_function=smoothing
        )
        return round(score, 4)
    except Exception:
        return 0.0

def count_prompt_tokens(text):
    """Approximate token count — words + punctuation"""
    return len(re.findall(r'\w+|[^\w\s]', text))

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
from huggingface_hub import login
login()

In [3]:
# Cell 2 — Load model
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "neo4j/text-to-cypher-Gemma-3-4B-Instruct-2025.04.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded!")
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

Model loaded!
Memory: 0.14 GB


In [38]:
# Cell 3 — Neo4j + Groq connections
import os
from neo4j import GraphDatabase
from groq import Groq




driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
groq_client = Groq(api_key=GROQ_API_KEY)
EXTRACTOR_MODEL = "llama-3.1-8b-instant"

In [14]:

# Cell 4 — Schema + graph values
# SCHEMA = """Node properties:
# - Customer
#   - customer_id: STRING Example: "12345abc"
#   - city: STRING Example: "sao paulo"
#   - state: STRING Example: "SP"
# - Order
#   - order_id: STRING Example: "abc123"
#   - status: STRING Example: "delivered" (possible values: delivered, canceled, shipped, processing, invoiced)
#   - purchase_date: STRING Example: "2017-10-02 10:56:33"
# - Product
#   - product_id: STRING Example: "abc123"
#   - category: STRING Example: "esporte_lazer"
# - Seller
#   - seller_id: STRING Example: "abc123"
#   - city: STRING Example: "sao paulo"
#   - state: STRING Example: "SP"
# - Review
#   - review_id: STRING Example: "abc123"
#   - score: INTEGER Min: 1, Max: 5

# Relationship properties:
# The relationships:
# (:Customer)-[:PLACED]->(:Order)
# (:Order)-[:CONTAINS]->(:Product)
# (:Product)-[:SOLD_BY]->(:Seller)
# (:Order)-[:HAS_REVIEW]->(:Review)"""

SCHEMA = """Node properties:
- Customer
  - customer_id: STRING Example: "12345abc"
  - city: STRING Example: "sao paulo"
  - state: STRING Example: "SP"
- Order
  - order_id: STRING Example: "abc123"
  - status: STRING Example: "delivered" (possible values: delivered, canceled, shipped, processing, invoiced)
  - purchase_date: STRING Example: "2017-10-02 10:56:33"
- Product
  - product_id: STRING Example: "abc123"
  - category: STRING Example: "esporte_lazer"
- Seller
  - seller_id: STRING Example: "abc123"
  - city: STRING Example: "sao paulo"
  - state: STRING Example: "SP"
- Review
  - review_id: STRING Example: "abc123"
  - score: INTEGER Min: 1, Max: 5

Relationship properties:
The relationships:
(:Customer)-[:PLACED]->(:Order)
(:Order)-[:CONTAINS]->(:Product)
(:Product)-[:HAS_SELLER]->(:Seller)
(:Order)-[:HAS_REVIEW]->(:Review)"""

def get_values(query):
    with driver.session() as session:
        return [list(r.values())[0] for r in session.run(query).data()]

GRAPH_VALUES = {
    "Customer.state":   get_values("MATCH (c:Customer) RETURN DISTINCT c.state AS v ORDER BY v"),
    "Customer.city":    get_values("MATCH (c:Customer) RETURN DISTINCT c.city AS v ORDER BY v LIMIT 50"),
    "Order.status":     get_values("MATCH (o:Order) RETURN DISTINCT o.status AS v"),
    "Product.category": get_values("MATCH (p:Product) RETURN DISTINCT p.category AS v ORDER BY v"),
    "Seller.state":     get_values("MATCH (s:Seller) RETURN DISTINCT s.state AS v ORDER BY v"),
    "Seller.city":      get_values("MATCH (s:Seller) RETURN DISTINCT s.city AS v ORDER BY v LIMIT 30"),
    "Review.score":     get_values("MATCH (r:Review) RETURN DISTINCT r.score AS v ORDER BY v"),
}
print("Graph values loaded!")

Graph values loaded!


In [49]:
FULL_SCHEMA_DESCRIPTION = """Available nodes and relationships:
Nodes: Customer, Order, Product, Seller, Review
Relationships:
- (Customer)-[:PLACED]->(Order)
- (Order)-[:CONTAINS]->(Product)
- (Product)-[:SOLD_BY]->(Seller)
- (Order)-[:HAS_REVIEW]->(Review)"""

def prune_schema(question):
    prompt = f"""You are a schema selector for a Neo4j e-commerce knowledge graph.

{FULL_SCHEMA_DESCRIPTION}

Given the question, return ONLY the nodes and relationships needed to answer it.
Return a JSON object with two keys:
- "nodes": list of node names needed
- "relationships": list of relationships needed, each as "(:NodeA)-[:REL]->(:NodeB)"

Examples:
Question: "How many products has each seller sold?"
Output: {{"nodes": ["Product", "Seller"], "relationships": ["(:Product)-[:SOLD_BY]->(:Seller)"]}}

Question: "{question}"
Output:"""

    response = groq_client.chat.completions.create(
        model=EXTRACTOR_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        temperature=0.0
    )
    raw = response.choices[0].message.content.strip()
    try:
        start = raw.find("{")
        end   = raw.rfind("}") + 1
        if start != -1 and end > 0:
            parsed = json.loads(raw[start:end])
            return parsed
    except Exception:
        pass
    return None  # fallback to full schema

NODE_PROPERTIES = {
    "Customer": '- Customer\n  - customer_id: STRING Example: "12345abc"\n  - city: STRING Example: "sao paulo"\n  - state: STRING Example: "SP"',
    "Order":    '- Order\n  - order_id: STRING Example: "abc123"\n  - status: STRING Example: "delivered" (possible values: delivered, canceled, shipped, processing, invoiced)\n  - purchase_date: STRING Example: "2017-10-02 10:56:33"',
    "Product":  '- Product\n  - product_id: STRING Example: "abc123"\n  - category: STRING Example: "esporte_lazer"',
    "Seller":   '- Seller\n  - seller_id: STRING Example: "abc123"\n  - city: STRING Example: "sao paulo"\n  - state: STRING Example: "SP"',
    "Review":   '- Review\n  - review_id: STRING Example: "abc123"\n  - score: INTEGER Min: 1, Max: 5',
}

def build_pruned_schema(question):
    result = prune_schema(question)
    
    if result is None:
        return SCHEMA  # fallback to full schema
    
    nodes     = result.get("nodes", [])
    rels      = result.get("relationships", [])
    
    if not nodes:
        return SCHEMA  # fallback
    
    nodes_str = "Node properties:\n" + "\n".join(
        NODE_PROPERTIES[n] for n in nodes if n in NODE_PROPERTIES
    )
    rels_str  = "The relationships:\n" + "\n".join(rels) if rels else ""
    
    return f"{nodes_str}\n\nRelationship properties:\n{rels_str}"

In [71]:

# # Cell 5 — Text2Cypher generation using Neo4j model
# def generate_cypher_neo4j_model(question, entity_str=""):
#     content = (
#         f"Generate Cypher statement to query a graph database.\n"
#         f"Use ONLY the provided relationship types and properties in the schema.\n\n"
#         f"Schema:\n{SCHEMA}\n"
#         f"{entity_str}\n"
#         f"Question: {question}\n"
#         f"Cypher output:"
#     )
#     messages = [{"role": "user", "content": content}]
    
#     # fix: tokenize separately
#     prompt = tokenizer.apply_chat_template(
#         messages,
#         add_generation_prompt=True,
#         tokenize=False
#     )
#     input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

#     with torch.inference_mode():
#         outputs = model.generate(
#             input_ids,
#             max_new_tokens=200,
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id
#         )
#     new_tokens = outputs[0][input_ids.shape[-1]:]
#     raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
#     return postprocess(raw)

def generate_cypher_neo4j_model(question, use_pruning=False):
    schema = build_pruned_schema(question) if use_pruning else SCHEMA

    content = f"""Generate Cypher statement to query a graph database.
Use ONLY the provided relationship types and properties in the schema.

Schema:
{schema}

Question: {question}
Cypher:"""

    messages  = [{"role": "user", "content": content}]
    prompt    = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][input_ids.shape[-1]:]
    raw        = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return postprocess(raw), content  # return both cypher and prompt

In [75]:
# Cell 7 — Postprocess + execute
def postprocess(text):
    text = text.replace('\\n', '\n').strip()
    if "```" in text:
        parts = text.split("```")
        if len(parts) >= 3:
            inner = parts[1]
            if inner.startswith("cypher"):
                inner = inner[len("cypher"):]
            text = inner.strip()
    for marker in ["**Explanation", "Explanation:", "Note:", "This query"]:
        if marker in text:
            text = text[:text.index(marker)].strip()
    # rename back to actual graph relationship
    text = text.replace("HAS_SELLER", "SOLD_BY")
    return text.strip()


def execute_cypher(cypher):
    try:
        with driver.session() as session:
            return session.run(cypher).data()
    except Exception:
        return None

def results_match(gold, pred):
    if pred is None:
        return False
    try:
        if len(gold) != len(pred):
            return False
        
        def get_values(row):
            return set(str(v) for v in row.values())
        
        gold_sorted = sorted(gold, key=lambda x: str(sorted(x.items())))
        pred_sorted = sorted(pred, key=lambda x: str(sorted(x.items())))
        
        for g_row, p_row in zip(gold_sorted, pred_sorted):
            gold_values = get_values(g_row)
            pred_values = get_values(p_row)
            # gold values must be subset of predicted values
            if not gold_values.issubset(pred_values):
                return False
        return True
    except Exception:
        return False

In [68]:
test_questions = [
    "What is the average review score across all orders?",
]

for q in test_questions:
    print(f"Q: {q}")
    result = prune_schema(q)
    print(f"Pruned: {result}\n")

Q: What is the average review score across all orders?
Pruned: {'nodes': ['Order', 'Review'], 'relationships': ['(:Order)-[:HAS_REVIEW]->(:Review)']}



In [73]:
test_cases = [
    "What is the average review score across all orders?",      # single hop SOLD_BY
]
for q in test_cases:
    cypher = generate_cypher_neo4j_model(q,use_pruning=True)
    print(f"Q: {q}")
    print(f"Generated: {cypher}\n")

Q: What is the average review score across all orders?
Generated: ('MATCH (:Order)-[r:HAS_REVIEW]->(review:Review)\nWITH avg(review.score) AS average_score\nRETURN average_score', 'Generate Cypher statement to query a graph database.\nUse ONLY the provided relationship types and properties in the schema.\n\nSchema:\nNode properties:\n- Order\n  - order_id: STRING Example: "abc123"\n  - status: STRING Example: "delivered" (possible values: delivered, canceled, shipped, processing, invoiced)\n  - purchase_date: STRING Example: "2017-10-02 10:56:33"\n- Review\n  - review_id: STRING Example: "abc123"\n  - score: INTEGER Min: 1, Max: 5\n\nRelationship properties:\nThe relationships:\n(:Order)-[:HAS_REVIEW]->(:Review)\n\nQuestion: What is the average review score across all orders?\nCypher:')



In [77]:
import time, json, re
from collections import defaultdict

def evaluate_full():
    with open("/kaggle/input/datasets/amitkumarkumavat/datagoldenpairs1/test_dataset.json") as f:
        dataset = json.load(f)

    systems = {
    "schema_aware": lambda q: generate_cypher_neo4j_model(q,  use_pruning=False),
    "with_pruning": lambda q: generate_cypher_neo4j_model(q, use_pruning=True),
    }

    results = {name: [] for name in systems}

    for i, item in enumerate(dataset):
        print(f"\n[{i+1}/50] Q{item['id']} ({item['complexity']}): {item['question'][:60]}...")
        gold_result = execute_cypher(item["gold_cypher"])

        for sys_name, gen_fn in systems.items():
            try:
                cypher, prompt_text = gen_fn(item["question"])
                pred_result  = execute_cypher(cypher)
                match        = results_match(gold_result, pred_result)
                bleu         = compute_google_bleu(item["gold_cypher"], cypher)
                token_count  = count_prompt_tokens(prompt_text) if prompt_text else 0

                results[sys_name].append({
                    "id":               item["id"],
                    "complexity":       item["complexity"],
                    "question":         item["question"],
                    "generated_cypher": cypher,
                    "executable":       pred_result is not None,
                    "correct":          match,
                    "bleu":             bleu,
                    "token_count":      token_count,
                })
                status = "✓" if match else ("✗ not exec" if pred_result is None else "✗ wrong")
                print(f"  {sys_name:<20}: {status} | BLEU={bleu:.3f} | tokens={token_count}")

            except Exception as e:
                print(f"  {sys_name:<20}: ERROR — {e}")
                results[sys_name].append({
                    "id": item["id"], "complexity": item["complexity"],
                    "question": item["question"], "generated_cypher": "",
                    "executable": False, "correct": False, "bleu": 0.0, "token_count": 0
                })

        time.sleep(0.5)

    # save raw results
    with open("/kaggle/working/results_final.json", "w") as f:
        json.dump(results, f, indent=2)

    # print summary
    print_summary(results, dataset)


def print_summary(results, dataset):
    print("\n\n══════════════════════════════════════════════════════════")
    print("  FINAL RESULTS — Neo4j Gemma-3-4B")
    print("══════════════════════════════════════════════════════════")
    print(f"  {'System':<22} {'EX%':>6}  {'BLEU':>6}  {'Avg Tokens':>10}")
    print("  " + "─"*50)

    for sys_name, res in results.items():
        ex       = sum(r["correct"] for r in res) / len(res) * 100
        bleu     = sum(r["bleu"] for r in res) / len(res)
        tokens   = sum(r["token_count"] for r in res) / len(res)
        print(f"  {sys_name:<22} {ex:>5.1f}%  {bleu:>6.4f}  {tokens:>10.1f}")

    print("\n  Breakdown by complexity:")
    for complexity in ["simple", "filter", "aggregation", "multihop"]:
        print(f"\n  [{complexity}]")
        for sys_name, res in results.items():
            subset = [r for r in res if r["complexity"] == complexity]
            if subset:
                ex    = sum(r["correct"] for r in subset) / len(subset) * 100
                bleu  = sum(r["bleu"] for r in subset) / len(subset)
                tok   = sum(r["token_count"] for r in subset) / len(subset)
                print(f"    {sys_name:<22}: EX={ex:.1f}% BLEU={bleu:.4f} Tokens={tok:.1f} ({sum(r['correct'] for r in subset)}/{len(subset)})")

    print("\n══════════════════════════════════════════════════════════")

In [78]:
evaluate_full()


[1/50] Q1 (simple): How many customers are there in total?...
  schema_aware        : ✓ | BLEU=1.000 | tokens=258
  with_pruning        : ✓ | BLEU=1.000 | tokens=71

[2/50] Q2 (simple): How many orders have been placed?...
  schema_aware        : ✓ | BLEU=0.176 | tokens=257
  with_pruning        : ✓ | BLEU=0.421 | tokens=110

[3/50] Q3 (simple): How many sellers are there?...
  schema_aware        : ✓ | BLEU=0.912 | tokens=256
  with_pruning        : ✓ | BLEU=0.470 | tokens=69

[4/50] Q4 (simple): How many products are in the graph?...
  schema_aware        : ✓ | BLEU=0.912 | tokens=258
  with_pruning        : ✓ | BLEU=0.912 | tokens=61

[5/50] Q5 (simple): List all unique order statuses....
  schema_aware        : ✓ | BLEU=0.834 | tokens=256
  with_pruning        : ✓ | BLEU=0.834 | tokens=91

[6/50] Q6 (simple): List all unique product categories....
  schema_aware        : ✓ | BLEU=0.834 | tokens=256
  with_pruning        : ✓ | BLEU=0.375 | tokens=59

[7/50] Q7 (simple): How many re

In [79]:
with open("/kaggle/working/results_final.json") as f:
    data = json.load(f)

print("=== FILTER FAILURES WITH PRUNING ===")
for r in data["with_pruning"]:
    if r["complexity"] == "aggregation" and not r["correct"]:
        print(f"Q{r['id']}: {r['question']}")
        print(f"Generated: {r['generated_cypher']}\n")

=== FILTER FAILURES WITH PRUNING ===
Q25: How many orders does each seller have?
Generated: MATCH (s:Seller)-[:SOLD_BY]->()
WITH s, count{(s)-[:SOLD_BY]->()} AS orderCount
RETURN s.seller_id, orderCount

Q27: What is the average review score per product category?
Generated: MATCH (o:Order)-[:HAS_REVIEW]->(r:Review)-[:CONTAINS]->(p:Product)
WITH p.category AS category, avg(r.score) AS avg_score
RETURN category, avg_score

Q28: How many orders were placed per state?
Generated: MATCH (c:Customer)-[:PLACED]->(o:Order)
RETURN c.state AS state, COUNT(o) AS order_count
ORDER BY state

Q29: Which seller has sold products in the most orders?
Generated: MATCH (s:Seller)-[:SOLD_BY]->(p:Product)<-[:CONTAINS]-(o:Order)
WITH s, COUNT(o) AS order_count
ORDER BY order_count DESC
LIMIT 1
RETURN s.seller_id, order_count

Q30: How many products has each seller sold?
Generated: MATCH (s:Seller)-[:SOLD_BY]->(p:Product)
RETURN s.seller_id, COUNT(p) AS products_sold

